<a href="https://vigneashpandiyan.github.io/publications/Codes/" target="_blank" rel="noopener noreferrer">
  <img src="https://vigneashpandiyan.github.io/images/Link.png"
       style="max-width: 800px; width: 100%; height: auto;">
</a>

# Activation Functions

In [ ]:
import json
import math
import os
import urllib.request
import warnings
from urllib.error import HTTPError

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torchvision

# %matplotlib inline
from IPython.display import set_matplotlib_formats
from torchvision import transforms
from torchvision.datasets import FashionMNIST
from tqdm.notebook import tqdm

set_matplotlib_formats("svg", "pdf")  # For export
sns.set()

In [ ]:
# Path to the folder where the datasets are/should be downloaded (e.g. MNIST)
DATASET_PATH = os.environ.get("PATH_DATASETS", "data/")
# Path to the folder where the pretrained models are saved
CHECKPOINT_PATH = os.environ.get("PATH_CHECKPOINT", "saved_models/Activation_Functions/")


# Function for setting the seed
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():  # GPU operation have separate seed
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_seed(42)

# Additionally, some operations on a GPU are implemented stochastic for efficiency
# We want to ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.determinstic = True
torch.backends.cudnn.benchmark = False

# Fetching the device that will be used throughout this notebook
device = torch.device("cpu") if not torch.cuda.is_available() else torch.device("cuda:0")
print("Using device", device)

## Mathematical Formulas

Activation functions introduce non-linearity to neural networks, allowing them to learn complex patterns. Below are the mathematical definitions for the functions we will implement.

### **1. Sigmoid**
Squashes numbers into the range $(0, 1)$. It is often used for binary classification probabilities.
$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

### **2. Tanh (Hyperbolic Tangent)**
Squashes numbers into the range $(-1, 1)$. It is zero-centered, making it generally preferred over Sigmoid for hidden layers.
$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$$

### **3. ReLU (Rectified Linear Unit)**
The most common activation function. It returns $x$ if positive, else $0$. It is computationally efficient but can suffer from "dead neurons" if gradients fall to 0.
$$f(x) = \max(0, x)$$

### **4. Leaky ReLU**
A variant of ReLU that allows a small gradient $\alpha$ when $x \le 0$ (typically $\alpha=0.01$). This prevents neurons from "dying."
$$f(x) = \begin{cases} x & \text{if } x > 0 \\ \alpha x & \text{if } x \le 0 \end{cases}$$

### **5. ELU (Exponential Linear Unit)**
Uses a log curve for negative values to smooth the transition and push mean activation closer to zero.
$$f(x) = \begin{cases} x & \text{if } x > 0 \\ \alpha (e^x - 1) & \text{if } x \le 0 \end{cases}$$

### **6. Swish**
A self-gated activation function discovered by Google researchers. It often performs better than ReLU in very deep networks.
$$f(x) = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}}$$

### Quick Reference Table

| Function | Output Range | Analogy | Best Use Case |
| :--- | :--- | :--- | :--- |
| **ReLU** | $[0, \infty)$ | **Positive Filter** | Default for Hidden Layers |
| **Sigmoid** | $[0, 1]$ | **Probability Meter** | Output Layer (Binary Classification) |
| **Tanh** | $[-1, 1]$ | **Balanced Scale** | Hidden Layers (RNNs) |
| **LeakyReLU**| $(-\infty, \infty)$ | **Leaky Faucet** | If ReLU neurons are "dying" |
| **Softmax** | $[0, 1]$ (Sums to 1) | **Pie Chart** | Output Layer (Multi-class) |

### Visualizing Before and After Activation

Let's visualize the effect of these activation functions on a sample of input data. We'll generate some random input values and then apply each activation function to them, plotting the original values against the activated values.

In [ ]:
num_samples = 1000
input_data = torch.randn(num_samples) * 5 # Generate random data, scaled for better visualization
input_data = input_data.sort().values # Sort for a smoother plot

# --- Start of added code to resolve NameError: act_fns not defined ---
# These definitions are included here to make this cell self-contained,
# as the original definitions are in other cells that may not have been executed.
import torch.nn as nn

class ActivationFunction(nn.Module):
    def __init__(self):
        super().__init__()
        self.name = self.__class__.__name__
        self.config = {"name": self.name}

class Sigmoid(ActivationFunction):
    def forward(self, x):
        return 1 / (1 + torch.exp(-x))


class Tanh(ActivationFunction):
    def forward(self, x):
        x_exp, neg_x_exp = torch.exp(x), torch.exp(-x)
        return (x_exp - neg_x_exp) / (x_exp + neg_x_exp)


class ReLU(ActivationFunction):
    def forward(self, x):
        return x * (x > 0).float()


class LeakyReLU(ActivationFunction):
    def __init__(self, alpha=0.1):
        super().__init__()
        self.config["alpha"] = alpha

    def forward(self, x):
        return torch.where(x > 0, x, self.config["alpha"] * x)


class ELU(ActivationFunction):
    def forward(self, x):
        return torch.where(x > 0, x, torch.exp(x) - 1)


class Swish(ActivationFunction):
    def forward(self, x):
        return x * torch.sigmoid(x)




In [ ]:
act_fn_by_name = {"sigmoid": Sigmoid, "tanh": Tanh, "relu": ReLU, "leakyrelu": LeakyReLU, "elu": ELU, "swish": Swish}
act_fns = [act_fn() for act_fn in act_fn_by_name.values()]


In [ ]:
# Prepare plotting
cols = 2
rows = math.ceil(len(act_fns) / float(cols))
fig, ax = plt.subplots(rows, cols, figsize=(cols * 6, rows * 5))
fig.suptitle('Before and After Activation Function Application', fontsize=16)

# Set figure background to white
fig.patch.set_facecolor('white')

# Get a color palette for the activation functions
colors = sns.color_palette('tab10', len(act_fns)) # Using 'tab10' for distinct colors

for i, act_fn in enumerate(act_fns):
    # Apply activation function
    output_data = act_fn(input_data)

    # Convert to numpy for plotting
    x_np = input_data.cpu().numpy()
    y_np = output_data.cpu().numpy()

    # Get the correct subplot axis
    current_ax = ax[divmod(i, cols)] if rows > 1 else ax[i]

    # Set subplot background to white
    current_ax.set_facecolor('white')

    current_ax.plot(x_np, x_np, '--', color='gray', label='Input (y=x)') # Plot identity line
    current_ax.plot(x_np, y_np, linewidth=2, label=f'Output ({act_fn.name})', color=colors[i]) # Assign distinct color
    current_ax.set_title(act_fn.name)
    current_ax.set_xlabel('Input')
    current_ax.set_ylabel('Output')
    current_ax.legend()
    current_ax.grid(False)

fig.subplots_adjust(hspace=0.4, wspace=0.3)
plt.show()

In [ ]:
def get_grads(act_fn, x):
    """This function computes the gradients of an activation function at specified positions.

    Args:
        act_fn: An object of the class "ActivationFunction" with an implemented forward pass.
        x: 1D input tensor.
    Returns:
        A tensor with the same size of x containing the gradients of act_fn at x.
    """
    x = x.clone().requires_grad_()  # Mark the input as tensor for which we want to store gradients
    out = act_fn(x)
    out.sum().backward()  # Summing results in an equal gradient flow to each element in x
    return x.grad  # Accessing the gradients of x by "x.grad"

## Visualising their Gradients

In [ ]:
def vis_act_fn(act_fn, ax, x, color):
    ax.set_facecolor('white') # Set subplot background to white
    # Run activation function
    y = act_fn(x)
    y_grads = get_grads(act_fn, x)
    # Push x, y and gradients back to cpu for plotting
    x, y, y_grads = x.cpu().numpy(), y.cpu().numpy(), y_grads.cpu().numpy()
    # Plotting
    ax.plot(x, y, linewidth=2, label="ActFn", color=color) # Use passed color
    ax.plot(x, y_grads, linewidth=2, label="Gradient", color='darkgray', linestyle=':') # Give gradient a distinct but subtle color
    ax.set_title(act_fn.name)
    ax.legend()
    ax.set_ylim(-1.5, x.max())
    ax.grid(False) # Ensure no grid lines


# Add activation functions if wanted
act_fns = [act_fn() for act_fn in act_fn_by_name.values()]
x = torch.linspace(-5, 5, 1000)  # Range on which we want to visualize the activation functions
# Plotting
cols = 2
rows = math.ceil(len(act_fns) / float(cols))
fig, ax = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
fig.patch.set_facecolor('white') # Set figure background to white
colors = sns.color_palette('tab10', len(act_fns)) # Get color palette

for i, act_fn in enumerate(act_fns):
    vis_act_fn(act_fn, ax[divmod(i, cols)], x, colors[i]) # Pass color to vis_act_fn
fig.subplots_adjust(hspace=0.3)
plt.show()

## Effect of activation functions on some numerical values:

Below can observed the effect of the activation functions on a range of input values. Notice the variation in output range for each function.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import math # Import math for ceil function

# 1. Define the input matrix (3x3)
# Intentionally using a mix of negative, positive, and large values
input_data = torch.tensor(
    [[-3, -1, -0.7], [-0.2, 0, 0.2], [0.7, 1.0, 3.0]]
)


act_fn_by_name = {
    "sigmoid": Sigmoid,
    "tanh": Tanh,
    "relu": ReLU,
    "leakyrelu": LeakyReLU,
    "elu": ELU,
    "swish": Swish,
}
act_fns = [act_fn() for act_fn in act_fn_by_name.values()]

# 2. Setup the Plotting Grid
n_cols = 1
total_plots = len(act_fns) + 1 # +1 for the original input data
n_rows = math.ceil(total_plots / float(n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5, n_rows * 1.5))
axes = axes.flatten()  # Flatten 2D array of axes to 1D for easy looping

# 3. Plot the Original Input Data First
ax = axes[0]
sns.heatmap(input_data.numpy(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, cbar=False)
ax.set_title('Original Input', fontsize=14, fontweight='bold')
ax.axis('off')  # Hide x and y axis ticks for cleaner look

# 4. Loop through each function and plot their outputs
for i, act_fn_instance in enumerate(act_fns):
    ax = axes[i + 1] # Start from the second subplot

    # Apply the activation function
    output_matrix = act_fn_instance(input_data) # Call the instance with input_data
    data = output_matrix.numpy()

    # Create a Heatmap
    # annot=True adds the numbers to the squares
    # cmap="coolwarm" makes negatives blue and positives red
    sns.heatmap(data, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, cbar=False)

    ax.set_title(act_fn_instance.name, fontsize=14, fontweight='bold') # Use instance name for title
    ax.axis('off')  # Hide x and y axis ticks for cleaner look

# Hide any unused subplots
for i in range(total_plots, len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

# Known Drawbacks:

### The Vanishing Gradient Problem

The vanishing gradient problem is a common challenge when training deep neural networks, especially those using activation functions like Sigmoid or Tanh. During backpropagation, gradients are multiplied layer by layer. If these gradients are consistently small (e.g., less than 1), they can shrink exponentially as they propagate backward through many layers, eventually becoming negligible for the earlier layers. This makes it very difficult for the weights in the initial layers to update and learn effectively.

This code demonstrates the vanishing gradient problem by:

1.  **Constructing a Deep Network**: It builds a simple deep neural network (10 layers) where each layer consists of a linear transformation followed by an activation function.
2.  **Simulating Backpropagation**: It performs a forward pass with dummy input data and then a backward pass (backpropagation) to compute gradients for all layer weights.
3.  **Measuring Gradient Magnitude**: For each layer, it captures and measures the average absolute magnitude of the gradients associated with its weights.

By comparing the gradient magnitudes across layers for both `Sigmoid` and `ReLU` activation functions, we can visualize:
*   How gradients tend to shrink rapidly and 'vanish' in deeper layers when using `Sigmoid`.
*   How `ReLU` (Rectified Linear Unit) helps to mitigate this problem by having a constant, non-zero gradient (for positive inputs), allowing gradients to flow more effectively through deeper layers and facilitating better learning.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

def test_vanishing_gradient(activation_fn, label):
    # Create a deep network (10 layers) with the chosen activation
    layers = []
    for _ in range(10):
        layers.append(nn.Linear(10, 10))
        layers.append(activation_fn)
    model = nn.Sequential(*layers)

    # Fake input and dummy loss
    input_data = torch.randn(1, 10)
    output = model(input_data)
    loss = output.mean()
    loss.backward()

    # Capture gradients from each layer
    grads = []
    for name, param in model.named_parameters():
        if "weight" in name and param.grad is not None:
            grads.append(param.grad.abs().mean().item())

    return grads

# Run experiment
sigmoid_grads = test_vanishing_gradient(nn.Sigmoid(), "Sigmoid")
relu_grads = test_vanishing_gradient(nn.ReLU(), "ReLU")

# Plot
plt.figure(figsize=(10, 5))
plt.plot(sigmoid_grads, label='Sigmoid Gradients', marker='o')
plt.plot(relu_grads, label='ReLU Gradients', marker='x')
plt.title("The Vanishing Gradient Problem")
plt.xlabel("Layer Depth (0 = First Layer, 9 = Last Layer)")
plt.ylabel("Average Gradient Magnitude")
plt.yscale('log') # Log scale to show the massive difference
plt.legend()
plt.grid(True, which="both", ls="--")
plt.show()

### The "Dead Neuron" Problem (with ReLU)

One common issue with the ReLU (Rectified Linear Unit) activation function is the "dead neuron" problem. This occurs when a neuron's output is consistently zero for *all* inputs, effectively making it permanently inactive and unable to learn. This can happen if the weights driving the neuron are updated in such a way that the weighted sum of inputs always results in a negative value, pushing the neuron into the zero-gradient region of ReLU.

This code simulates a scenario to visualize this problem:

1.  **Simulating a Layer**: It creates a simple linear layer representing a neural network layer.
2.  **Input Scaling**: It introduces `input_scale` to simulate different input distributions. A high `input_scale` can mimic scenarios where weights grow large, potentially leading to more negative inputs for ReLU neurons.
3.  **Counting Dead Neurons**: For each activation function, it counts the number of neurons that output zero, indicating they are "dead" or inactive.

By comparing ReLU with a normal input scale, ReLU with a high input scale, and LeakyReLU with a high input scale, we aim to illustrate:
*   How the ReLU activation can lead to a significant number of dead neurons, especially under certain conditions (like large negative inputs).
*   How LeakyReLU, by allowing a small gradient for negative inputs, can prevent neurons from becoming entirely inactive, thus mitigating the "dead neuron" problem.

In [ ]:
def check_dead_neurons(activation_fn, input_scale):
    # Simulate a hidden layer with 1000 neurons
    layer = nn.Linear(100, 1000)
    act = activation_fn

    # Create random input data
    # If we increase input_scale, we simulate "bad initialization" or "high learning rate"
    x = torch.randn(128, 100) * input_scale

    # Forward pass
    with torch.no_grad():
        out = act(layer(x))

    # Count zeros
    num_zeros = (out == 0).sum().item()
    total_elements = out.numel()
    dead_percentage = (num_zeros / total_elements) * 100

    print(f"Activation: {activation_fn.__class__.__name__}")
    print(f"Input Scale: {input_scale}")
    print(f"Dead/Inactive Neurons: {dead_percentage:.2f}%")
    print("-" * 30)

# Compare Normal vs High Variance initialization
check_dead_neurons(nn.ReLU(), input_scale=0.2)  # Normal
check_dead_neurons(nn.ReLU(), input_scale=20.0) # Simulate "exploding" weights -> High death rate
check_dead_neurons(nn.LeakyReLU(), input_scale=20.0) # Leaky ReLU saves them!

Note that usually there are a greater percentage of dead neurons in case 2 than case 1. However, due to the randomized datasets, there maybe exceptions. Therefore, run the block multiple times to note the statistics. You will see that case 3 using LeakyReLU consistently performs best (i.e. least number of dead neurons) even with high input scaling.

### Speed of Activation Functions

Beyond their mathematical properties, the computational speed of activation functions is crucial, especially in deep neural networks with many layers and parameters. Simpler functions like ReLU are generally faster to compute than more complex ones like Sigmoid or Tanh, which involve exponentials or divisions. This can significantly impact training and inference times. Below, we benchmark the speed of Sigmoid and ReLU on a large tensor to illustrate this difference.

In [ ]:
import time

def benchmark_activation(name, act_fn, size=(10000, 10000)):
    data = torch.randn(size)

    # Warmup (get the GPU/CPU ready)
    _ = act_fn(data)

    start = time.time()
    # Run 10 times to get an average
    for _ in range(50):
        _ = act_fn(data)
    end = time.time()

    print(f"{name}: {end - start:.4f} seconds")

print("--- Speed Benchmark (100 Million Parameters) ---")
benchmark_activation("Sigmoid (Math Heavy)", torch.sigmoid)
benchmark_activation("ReLU (Simple Math)", F.relu)

These seconds can add up to hours when training large models. This is just another example of what to look for in a particular activation function when building your own models.